# Chapter 13: Deploying scikit-learn Models in Production

## 📋 Summary

Training a model is only the beginning. Getting it into production — where it delivers real business value — requires serialization, versioning, scaling, monitoring, and automation. This chapter covers the full model deployment lifecycle using scikit-learn, with a focus on MLOps best practices.

Topics include model serialization with `joblib` and `pickle`, building scalable prediction services, managing the model lifecycle (versioning, retraining), setting up deployment pipelines, and monitoring deployed models for drift and degradation.

---

## 🧠 Theoretical Explanation

### Model Serialization
After training, a model must be **serialized** (saved to disk) so it can be loaded and used for inference without retraining:
- **`joblib`**: Preferred for scikit-learn models — handles large numpy arrays efficiently using memory mapping
- **`pickle`**: Python's built-in serialization — works but less efficient for large arrays
- **`skops`**: Newer, more secure serialization format for scikit-learn objects

### Production Pipelines
A deployment-ready pipeline should include **all preprocessing steps** alongside the model. This ensures:
1. Preprocessing is applied identically at inference time as during training
2. The entire pipeline can be versioned, saved, and loaded as a single artifact
3. No risk of preprocessing/model mismatch

### Model Drift and Monitoring
Models degrade in production because:
- **Data drift**: The distribution of input features changes over time
- **Concept drift**: The relationship between features and target changes
- **Label drift**: The distribution of the target variable changes

Monitoring strategies:
- Track prediction distributions over time
- Compare feature statistics (mean, std, min, max) to training baseline
- Set up performance alerts with holdout labels

### Model Versioning
Best practices:
- Save models with timestamps or version numbers
- Log training metadata (data version, hyperparameters, metrics)
- Use tools like MLflow or Weights & Biases for experiment tracking
- Keep a model registry to manage multiple versions

### Scaling for Production
- **Batch prediction**: Precompute predictions for all inputs offline
- **Real-time inference**: Serve predictions via REST API (Flask, FastAPI)
- **Parallel prediction**: Use `n_jobs=-1` for compatible estimators


## 13.1 Model Serialization with joblib and pickle

In [1]:
import joblib
import pickle
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Train a model
cancer = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, test_size=0.2, random_state=42
)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42))
])
pipeline.fit(X_train, y_train)
print(f'Model accuracy: {pipeline.score(X_test, y_test):.4f}')

# Save with joblib
joblib.dump(pipeline, 'model_v1.pkl')
print(f'Model saved: {os.path.getsize("model_v1.pkl")/1024:.1f} KB')

# Load and verify
loaded_model = joblib.load('model_v1.pkl')
print(f'Loaded model accuracy: {loaded_model.score(X_test, y_test):.4f}')

Model accuracy: 0.9649
Model saved: 318.8 KB
Loaded model accuracy: 0.9649


## 13.2 Model Versioning

In [2]:
import json
from datetime import datetime
import numpy as np

def save_model_with_metadata(model, model_name, metrics, params, version):
    """Save model artifact with accompanying metadata."""
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filename = f'{model_name}_v{version}_{timestamp}.pkl'
    
    # Save model
    joblib.dump(model, filename)
    
    # Save metadata
    metadata = {
        'model_name': model_name,
        'version': version,
        'timestamp': timestamp,
        'filename': filename,
        'metrics': metrics,
        'params': params
    }
    meta_filename = filename.replace('.pkl', '_metadata.json')
    with open(meta_filename, 'w') as f:
        json.dump(metadata, f, indent=2)
    
    print(f'Saved: {filename}')
    return filename, metadata

# Save our model with metadata
metrics = {
    'accuracy': round(pipeline.score(X_test, y_test), 4),
    'train_size': len(X_train),
    'test_size': len(X_test)
}
params = pipeline.get_params()

filename, meta = save_model_with_metadata(
    pipeline, 'cancer_classifier', metrics,
    {'n_estimators': 100, 'random_state': 42}, version='1.0'
)

print('\nMetadata:')
print(json.dumps({k: v for k, v in meta.items() if k != 'params'}, indent=2))

Saved: cancer_classifier_v1.0_20260313_123839.pkl

Metadata:
{
  "model_name": "cancer_classifier",
  "version": "1.0",
  "timestamp": "20260313_123839",
  "filename": "cancer_classifier_v1.0_20260313_123839.pkl",
  "metrics": {
    "accuracy": 0.9649,
    "train_size": 455,
    "test_size": 114
  }
}


## 13.3 Scaling for Batch Predictions

In [3]:
import numpy as np
import time
import pandas as pd

def batch_predict(model, X, batch_size=100):
    """Run predictions in batches to manage memory."""
    predictions = []
    n_batches = int(np.ceil(len(X) / batch_size))
    
    for i in range(n_batches):
        start = i * batch_size
        end = min(start + batch_size, len(X))
        batch = X[start:end]
        preds = model.predict(batch)
        predictions.extend(preds)
    
    return np.array(predictions)

# Test batch prediction
start = time.time()
batch_preds = batch_predict(pipeline, X_test, batch_size=50)
elapsed = time.time() - start

print(f'Batch predictions: {len(batch_preds)} samples in {elapsed:.4f}s')
print(f'Predictions match: {(batch_preds == pipeline.predict(X_test)).all()}')

Batch predictions: 114 samples in 0.0347s
Predictions match: True


## 13.4 Monitoring Model Drift

In [4]:
import matplotlib.pyplot as plt
import numpy as np

def compute_feature_stats(X, feature_names=None):
    """Compute baseline statistics for drift detection."""
    stats = {}
    for i in range(X.shape[1]):
        name = feature_names[i] if feature_names is not None else f'feature_{i}'
        stats[name] = {
            'mean': float(np.mean(X[:, i])),
            'std': float(np.std(X[:, i])),
            'min': float(np.min(X[:, i])),
            'max': float(np.max(X[:, i]))
        }
    return stats

# Save training baseline
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
cancer = load_breast_cancer()
X_train_c, X_test_c, _, _ = train_test_split(cancer.data, cancer.target, test_size=0.2, random_state=42)

baseline_stats = compute_feature_stats(X_train_c, list(cancer.feature_names))

# Simulate production data with drift (add noise)
X_production = X_test_c + np.random.normal(0, 2, X_test_c.shape)
production_stats = compute_feature_stats(X_production, list(cancer.feature_names))

# Detect drift: compare means
print('Feature Drift Detection (mean shift):')
for feat in list(baseline_stats.keys())[:5]:
    base_mean = baseline_stats[feat]['mean']
    prod_mean = production_stats[feat]['mean']
    base_std = baseline_stats[feat]['std']
    drift = abs(prod_mean - base_mean) / (base_std + 1e-8)
    flag = 'DRIFT' if drift > 1.0 else 'OK'
    print(f'  {feat[:30]:30s}: base={base_mean:.2f}, prod={prod_mean:.2f}, z={drift:.2f}  [{flag}]')


Feature Drift Detection (mean shift):
  mean radius                   : base=14.12, prod=14.34, z=0.06  [OK]
  mean texture                  : base=19.19, prod=19.66, z=0.11  [OK]
  mean perimeter                : base=91.88, prod=92.21, z=0.01  [OK]
  mean area                     : base=654.38, prod=656.96, z=0.01  [OK]
  mean smoothness               : base=0.10, prod=-0.01, z=7.93  [DRIFT]


## 13.5 Deployment Pipeline with Validation Threshold

In [5]:
def evaluate_and_deploy(model, X_val, y_val, threshold=0.90, model_path='production_model.pkl'):
    """
    Evaluate model on validation set.
    Deploy only if accuracy exceeds threshold.
    """
    accuracy = model.score(X_val, y_val)
    print(f'Validation accuracy: {accuracy:.4f}')
    print(f'Deployment threshold: {threshold}')
    
    if accuracy >= threshold:
        joblib.dump(model, model_path)
        print(f'✅ Model DEPLOYED to {model_path}')
        return True
    else:
        print(f'❌ Model NOT deployed — accuracy below threshold')
        return False

# Test deployment check
deployed = evaluate_and_deploy(pipeline, X_test, y_test, threshold=0.90)

# Simulate a bad model
from sklearn.dummy import DummyClassifier
bad_model = DummyClassifier(strategy='most_frequent')
bad_model.fit(X_train, y_train)
print('\n--- Bad model check ---')
evaluate_and_deploy(bad_model, X_test, y_test, threshold=0.90)

Validation accuracy: 0.9649
Deployment threshold: 0.9
✅ Model DEPLOYED to production_model.pkl

--- Bad model check ---
Validation accuracy: 0.6228
Deployment threshold: 0.9
❌ Model NOT deployed — accuracy below threshold


False

## 13.6 Simple REST API Skeleton (FastAPI)

In [6]:
# This shows the code structure for serving a model via FastAPI
# (Not executed here — requires running a web server)

api_code = '''
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import numpy as np

app = FastAPI(title="Cancer Classifier API")

# Load model at startup
model = joblib.load("production_model.pkl")

class PredictionRequest(BaseModel):
    features: list[float]  # 30 features for breast cancer dataset

class PredictionResponse(BaseModel):
    prediction: int
    probability: float
    label: str

@app.post("/predict", response_model=PredictionResponse)
def predict(request: PredictionRequest):
    X = np.array(request.features).reshape(1, -1)
    prediction = model.predict(X)[0]
    probability = model.predict_proba(X)[0].max()
    label = "malignant" if prediction == 0 else "benign"
    return PredictionResponse(
        prediction=int(prediction),
        probability=float(probability),
        label=label
    )

@app.get("/health")
def health():
    return {"status": "healthy"}

# Run with: uvicorn app:app --host 0.0.0.0 --port 8000
'''

print('FastAPI deployment skeleton:')
print(api_code)

# Save as a deployable script
with open('app.py', 'w') as f:
    f.write(api_code)
print('Saved to app.py')

FastAPI deployment skeleton:

from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import numpy as np

app = FastAPI(title="Cancer Classifier API")

# Load model at startup
model = joblib.load("production_model.pkl")

class PredictionRequest(BaseModel):
    features: list[float]  # 30 features for breast cancer dataset

class PredictionResponse(BaseModel):
    prediction: int
    probability: float
    label: str

@app.post("/predict", response_model=PredictionResponse)
def predict(request: PredictionRequest):
    X = np.array(request.features).reshape(1, -1)
    prediction = model.predict(X)[0]
    probability = model.predict_proba(X)[0].max()
    label = "malignant" if prediction == 0 else "benign"
    return PredictionResponse(
        prediction=int(prediction),
        probability=float(probability),
        label=label
    )

@app.get("/health")
def health():
    return {"status": "healthy"}

# Run with: uvicorn app:app --host 0.0.0.0 --port 8000

Saved to app

## 🔑 Key Takeaways

- **Always save the full pipeline** (preprocessing + model) — not just the model — to avoid preprocessing mismatches at inference time.
- Use `joblib` for scikit-learn models — it's more efficient than `pickle` for large numpy arrays.
- **Version your models** with timestamps and metadata (training data version, metrics, hyperparameters).
- Implement **validation thresholds** before deploying — only promote models that meet performance criteria.
- Monitor production data for **drift** by comparing feature statistics to training baseline.
- For serving predictions, use batch prediction for offline workloads and REST APIs (FastAPI, Flask) for real-time inference.
- MLOps tools like **MLflow**, **Kubeflow**, and **Weights & Biases** automate experiment tracking, model registry, and deployment pipelines.
